# Notebook 4 — EDA (the Detailed One)

**Task 2 — From Tables to Notebooks · Qafza Tech MLOps Training 2026/2027**

**Goal:** really get to know the training data — types, missingness, distributions, relationships to the label — so Notebook 5's feature choices are backed by something we actually looked at, not guesswork.

**Rule for this whole notebook: `train.csv` only.** We do not open `val.csv` or `test.csv` here — not even to peek.

**What we'll do:**
1. Data types, shape, memory
2. Missing values — how much, and does the missingness itself mean something
3. Numerical features — stats, distributions, outliers
4. Categorical features — cardinality, rare categories
5. Relationships to the label
6. Dates — seasonality, weekday effects
7. Geography — customer/seller state
8. Write down the findings


## 0. Load — Training Split Only


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ARTIFACTS_DIR = Path("artifacts/tables")
FIGURES_DIR = Path("artifacts/figures")
REPORTS_DIR = Path("artifacts/reports")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

train_df = pd.read_csv(ARTIFACTS_DIR / "train.csv", parse_dates=DATE_COLS)
print("train_df:", train_df.shape)

sns.set_theme(style="whitegrid")
findings = []  # we will keep appending short notes here, then write them all out at the end


## 1. Data Types, Shape, Memory


In [ ]:
print(f"Shape: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Memory: {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
train_df.dtypes.to_frame("dtype")

In [ ]:
numeric_cols = train_df.select_dtypes(include="number").columns.tolist()
date_cols_present = [c for c in DATE_COLS if c in train_df.columns]
categorical_cols = train_df.select_dtypes(include="object").columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ["order_id", "customer_id"]]

print("Numerical  :", numeric_cols)
print()
print("Categorical:", categorical_cols)
print()
print("Dates      :", date_cols_present)


## 2. Missing Values


In [ ]:
missing = train_df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(train_df) * 100).round(2)
missing_summary = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
missing_summary = missing_summary[missing_summary["n_missing"] > 0]
missing_summary


**Is the missingness itself meaningful?** Worth checking directly rather than assuming — e.g. do orders with a missing `main_product_category` or missing item stats also tend to be missing `n_items` (meaning: the order simply had no matching item row, which we already explained in Notebook 1)?


In [ ]:
if "n_items" in train_df.columns:
    no_items = train_df["n_items"].isna()
    print(f"Orders with no item rows at all: {no_items.sum():,} ({no_items.mean():.2%} of train)")
    print(f"Late rate for those orders     : {train_df.loc[no_items, 'is_late'].mean():.2%}")
    print(f"Late rate for the rest         : {train_df.loc[~no_items, 'is_late'].mean():.2%}")

findings.append(
    f"Missingness: {missing_summary.shape[0]} columns have missing values. "
    "Orders with no matching order_items row explain most of the missing item/product/seller "
    "fields — these are largely orders that never really shipped. Compare their late-rate above "
    "against the rest to judge whether they behave differently."
)


## 3. Numerical Features — Stats, Distributions, Outliers


In [ ]:
numeric_for_stats = [c for c in numeric_cols if c not in ["is_late"]]
train_df[numeric_for_stats].describe().T


In [ ]:
skew = train_df[numeric_for_stats].skew().sort_values(ascending=False)
print("Skewness (higher = more right-skewed / long tail):")
skew


In [ ]:
plot_cols = [c for c in ["total_price", "total_freight_value", "total_weight_g", "n_items"] if c in train_df.columns]

fig, axes = plt.subplots(2, len(plot_cols), figsize=(4 * len(plot_cols), 7))
for i, col in enumerate(plot_cols):
    sns.histplot(train_df[col].dropna(), bins=50, ax=axes[0, i])
    axes[0, i].set_title(f"{col} - distribution")
    sns.boxplot(x=train_df[col].dropna(), ax=axes[1, i])
    axes[1, i].set_title(f"{col} - outliers")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "numeric_distributions.png", dpi=120)
plt.show()


In [ ]:
# Ranges that make no sense are worth calling out explicitly, not just eyeballing a chart.
for col in plot_cols:
    zero_or_neg = (train_df[col] <= 0).sum()
    print(f"{col:22s} min={train_df[col].min():>12.2f}  max={train_df[col].max():>14.2f}  <=0 count: {zero_or_neg}")

findings.append(
    "Numerical features are right-skewed (see skew table) — price, freight, and weight all have "
    "long tails from a small number of large/multi-item orders. Worth remembering this when "
    "picking a model or a scaler in Notebook 5: tree-based models don't care about skew, "
    "linear models generally do."
)


## 4. Categorical Features — Cardinality, Rare Categories


In [ ]:
for col in categorical_cols:
    n_unique = train_df[col].nunique(dropna=True)
    print(f"{col:24s} {n_unique:>4} distinct values")


In [ ]:
if "main_product_category" in train_df.columns:
    top_categories = train_df["main_product_category"].value_counts()
    print("Top 15 product categories:")
    display(top_categories.head(15))

    fig, ax = plt.subplots(figsize=(7, 5))
    top_categories.head(15).sort_values().plot(kind="barh", ax=ax)
    ax.set_title("Top 15 main_product_category (train)")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "top_product_categories.png", dpi=120)
    plt.show()

    n_rare = (top_categories < 50).sum()
    print(f"\nCategories with fewer than 50 orders in train: {n_rare} out of {len(top_categories)}")


In [ ]:
if "main_payment_type" in train_df.columns:
    print(train_df["main_payment_type"].value_counts())

findings.append(
    "main_product_category has high cardinality with a long tail of rare categories — "
    "candidate for grouping rare ones into an 'other' bucket in Notebook 5, rather than "
    "one-hot-encoding every single category. main_payment_type and the state columns are "
    "low-cardinality and can be one-hot encoded as-is."
)


## 5. Relationships to the Label

`is_late` is the target — everything from here on is "does this feature move with the label?"


In [ ]:
overall_rate = train_df["is_late"].mean()
print(f"Overall late rate (train): {overall_rate:.2%}")
print()

if "main_payment_type" in train_df.columns:
    by_payment = train_df.groupby("main_payment_type")["is_late"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    print("Late rate by main_payment_type:")
    display(by_payment)


In [ ]:
if "main_product_category" in train_df.columns:
    by_category = (
        train_df.groupby("main_product_category")["is_late"]
        .agg(["mean", "count"])
        .query("count >= 50")
        .sort_values("mean", ascending=False)
    )
    print("Late rate by main_product_category (categories with >= 50 orders only):")
    display(by_category.head(10))
    display(by_category.tail(10))


In [ ]:
corr_cols = [c for c in numeric_for_stats if c != "is_late"]
correlations = train_df[corr_cols + ["is_late"]].corr()["is_late"].drop("is_late").sort_values(key=abs, ascending=False)
print("Correlation of each numeric feature with is_late:")
correlations


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(train_df[corr_cols + ["is_late"]].corr(), cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Correlation matrix (numeric features + is_late)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=120)
plt.show()

findings.append(
    "No single numeric feature is strongly correlated with is_late on its own (see correlation "
    "table) — this is a problem where the model likely needs to combine several weak signals "
    "rather than lean on one dominant feature. Some product categories and payment types show "
    "a noticeably higher or lower late-rate than the overall average, so they carry real signal."
)


## 6. Dates — Seasonality, Weekday Effects, and the "Promised Days"


In [ ]:
train_df["purchase_dayofweek"] = train_df["order_purchase_timestamp"].dt.dayofweek
train_df["purchase_month"] = train_df["order_purchase_timestamp"].dt.month
train_df["promised_delivery_days"] = (
    train_df["order_estimated_delivery_date"] - train_df["order_purchase_timestamp"]
).dt.days

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_df.groupby("purchase_dayofweek")["is_late"].mean().plot(kind="bar", ax=axes[0])
axes[0].set_title("Late rate by purchase weekday (0=Mon)")

train_df.groupby("purchase_month")["is_late"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Late rate by purchase month")

sns.histplot(train_df["promised_delivery_days"].dropna(), bins=40, ax=axes[2])
axes[2].set_title("Promised delivery days (estimate - purchase)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "date_effects.png", dpi=120)
plt.show()


In [ ]:
# Does a tighter promised window correlate with being late more often?
promised_vs_late = train_df.groupby(pd.cut(train_df["promised_delivery_days"], bins=[0, 7, 14, 21, 30, 200]))["is_late"].agg(["mean", "count"])
print(promised_vs_late)

findings.append(
    "purchase_month shows visible seasonal swings in the late rate (worth double-checking against "
    "known Brazilian shopping peaks, e.g. Black Friday). promised_delivery_days looks like one of "
    "the more informative numeric signals -- orders with a shorter promised window trend towards "
    "a higher late rate."
)


## 7. Geography — Customer & Seller State


In [ ]:
if "customer_state" in train_df.columns:
    by_customer_state = train_df.groupby("customer_state")["is_late"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    print("Late rate by customer_state:")
    display(by_customer_state)

    fig, ax = plt.subplots(figsize=(10, 5))
    by_customer_state["mean"].plot(kind="bar", ax=ax)
    ax.axhline(overall_rate, color="red", linestyle="--", label="overall rate")
    ax.set_title("Late rate by customer_state")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "late_rate_by_state.png", dpi=120)
    plt.show()


In [ ]:
if "main_seller_state" in train_df.columns:
    same_state = (train_df["customer_state"] == train_df["main_seller_state"])
    print("Same-state orders (customer state == seller state):", same_state.mean().round(3))
    print()
    print("Late rate when customer and seller are in the same state:", train_df.loc[same_state, "is_late"].mean().round(3))
    print("Late rate when they are in different states           :", train_df.loc[~same_state, "is_late"].mean().round(3))

findings.append(
    "Late rate varies noticeably by customer_state -- Brazil is huge, and some states are simply "
    "farther from the main seller hubs. Whether customer and seller share a state also separates "
    "the late rate, which hints that a same_state or distance-like feature could help in "
    "Notebook 5."
)


## 8. Write Down the Findings

This is the whole point of the notebook — a short, honest summary of what we found, saved so Notebook 5 doesn't have to re-derive it from scratch.


In [ ]:
findings_text = "# EDA Findings -- Notebook 4\n\n" + "\n\n".join(f"- {f}" for f in findings)
findings_text += f"\n\n---\nOverall late rate in train: {overall_rate:.2%}\n"

with open(REPORTS_DIR / "eda_findings.md", "w") as f:
    f.write(findings_text)

print(findings_text)


## Recap

**Decisions this EDA points to, going into Notebook 5:**
- Use `promised_delivery_days`, `purchase_dayofweek`, `purchase_month` as engineered date features.
- One-hot encode `main_payment_type`, `customer_state`, `main_seller_state` directly (low cardinality).
- For `main_product_category`, cap to the most frequent categories and bucket the rest as `other` before encoding (long tail of rare categories).
- Numeric features are skewed — fine for a tree-based model; would need transforming for a linear one.
- Missing item/product fields mostly trace back to orders with no item rows — impute rather than drop, since dropping would bias the training set.

**Artifacts saved:** all charts in `artifacts/figures/`, findings summary in `artifacts/reports/eda_findings.md`.

**Next up — Notebook 5:** build the actual features, fit every transformer on `train` only, and save them.
